# DevMind AI — Dataset Quality Audit & Debugging Format

Audit cleaned datasets and convert records into a consistent debugging-oriented instruction format. Do not fabricate missing explanations or fixes.

In [ ]:
!pip -q install pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, pandas as pd
BASE = Path('/content/drive/MyDrive/DevMindAI/datasets')
PROCESSED = BASE / 'processed'
AUDIT = BASE / 'audit'; AUDIT.mkdir(parents=True, exist_ok=True)
TRAIN_FILE = PROCESSED / 'train.jsonl'
if TRAIN_FILE.exists():
    df = pd.read_json(TRAIN_FILE, lines=True)
    print('Loaded records:', len(df)); display(df.head())
else: print('No processed train file found. Run Notebook 04 first.')

## 1. Basic quality metrics
Flags are review aids, not automatic proof that records are bad.

In [ ]:
if 'df' in globals() and len(df):
    required = ['instruction','input','response','language','source_dataset']
    for c in required:
        if c not in df.columns: df[c] = ''
    metrics = {'rows': len(df), 'missing_instruction': int(df['instruction'].fillna('').str.strip().eq('').sum()), 'missing_input': int(df['input'].fillna('').str.strip().eq('').sum()), 'missing_response': int(df['response'].fillna('').str.strip().eq('').sum()), 'unknown_language': int(df['language'].fillna('unknown').eq('unknown').sum()), 'short_response': int(df['response'].fillna('').str.len().lt(20).sum()), 'very_long_response': int(df['response'].fillna('').str.len().gt(20000).sum())}
    display(pd.DataFrame([metrics]))
    df['audit_flag'] = ''
    df.loc[df['instruction'].fillna('').str.strip().eq(''), 'audit_flag'] += 'missing_instruction;'
    df.loc[df['response'].fillna('').str.strip().eq(''), 'audit_flag'] += 'missing_response;'
    df.loc[df['response'].fillna('').str.len().lt(20), 'audit_flag'] += 'short_response;'
    df.loc[df['response'].fillna('').str.len().gt(20000), 'audit_flag'] += 'very_long_response;'
    display(df[df['audit_flag'].ne('')].head(30))
else: print('Run Notebook 04 and load data first.')

## 2. Tentative task classification
Rule-based labels are heuristics. Spot-check them before relying on category labels.

In [ ]:
def classify_task(text):
    t = str(text).lower()
    if any(x in t for x in ['traceback','stack trace','exception','runtime error']): return 'error_analysis'
    if any(x in t for x in ['fix this code','bug','debug','correct the code','error in this code']): return 'debugging'
    if any(x in t for x in ['test case','unit test','pytest','assert']): return 'testing'
    if any(x in t for x in ['explain','what does','how does']): return 'code_explanation'
    if any(x in t for x in ['implement','write a program','function to']): return 'code_generation'
    return 'general_programming'

if 'df' in globals() and len(df):
    df['task_category'] = df.apply(lambda r: classify_task(str(r['instruction'])+' '+str(r['input'])), axis=1)
    display(df['task_category'].value_counts().rename_axis('category').reset_index(name='count'))
    display(df.groupby(['source_dataset','task_category']).size().reset_index(name='count'))

## 3. Consistent training format
Source content is preserved. The formatter does not invent root causes, fixes, or test results.

In [ ]:
def format_example(row):
    instruction = str(row.get('instruction','')).strip()
    input_text = str(row.get('input','')).strip()
    response = str(row.get('response','')).strip()
    language = str(row.get('language','unknown')).strip()
    category = str(row.get('task_category','general_programming')).strip()
    prompt = f'Task category: {category}\nProgramming language: {language}\nTask: {instruction}'
    if input_text: prompt += f'\n\nCode, context, or error:\n{input_text}'
    return {'prompt': prompt, 'completion': response, 'metadata': {'language': language, 'task_category': category, 'source_dataset': str(row.get('source_dataset','unknown')), 'source_record_id': str(row.get('source_record_id',''))}}

if 'df' in globals() and len(df):
    formatted = [format_example(r) for _, r in df.iterrows() if str(r.get('response','')).strip()]
    print('Formatted examples:', len(formatted)); display(pd.DataFrame(formatted[:3]))

## 4. Export audit and formatted pilot
Review quality flags, category distribution, and samples manually before training.

In [ ]:
if 'df' in globals() and len(df):
    df.to_csv(AUDIT / 'quality_audit.csv', index=False)
    with (AUDIT / 'formatted_pilot.jsonl').open('w', encoding='utf-8') as f:
        for item in formatted: f.write(json.dumps(item, ensure_ascii=False)+'\n')
    print('Saved:', AUDIT / 'quality_audit.csv')
    print('Saved:', AUDIT / 'formatted_pilot.jsonl')
    print('Review these outputs manually before training.')

## Next
Notebook 06 will train a tokenizer on the reviewed training corpus only. Validation and test data remain held out.